In [72]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_moons, make_circles
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_graphviz, DecisionTreeRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, r2_score, mean_squared_error

In [73]:
import pandas as pd
obesidad = pd.read_csv("Obesidad.csv")
print(obesidad.columns)

Index(['YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc', 'Datasource',
       'Class', 'Topic', 'Question', 'Data_Value_Unit', 'Data_Value_Type',
       'Data_Value', 'Data_Value_Alt', 'Data_Value_Footnote_Symbol',
       'Data_Value_Footnote', 'Low_Confidence_Limit', 'High_Confidence_Limit ',
       'Sample_Size', 'Total', 'Age(years)', 'Education', 'Gender', 'Income',
       'Race/Ethnicity', 'GeoLocation', 'ClassID', 'TopicID', 'QuestionID',
       'DataValueTypeID', 'LocationID', 'StratificationCategory1',
       'Stratification1', 'StratificationCategoryId1', 'StratificationID1'],
      dtype='object')


In [74]:
obesidad.isna().sum()

YearStart                         0
YearEnd                           0
LocationAbbr                      0
LocationDesc                      0
Datasource                        0
Class                             0
Topic                             0
Question                          0
Data_Value_Unit               53392
Data_Value_Type                   0
Data_Value                     5046
Data_Value_Alt                 5046
Data_Value_Footnote_Symbol    48346
Data_Value_Footnote           48346
Low_Confidence_Limit           5046
High_Confidence_Limit          5046
Sample_Size                    5046
Total                         51485
Age(years)                    41954
Education                     45764
Gender                        49578
Income                        40043
Race/Ethnicity                38136
GeoLocation                    1008
ClassID                           0
TopicID                           0
QuestionID                        0
DataValueTypeID             

In [75]:
obesidad_recortado = obesidad[['YearStart','LocationAbbr','Topic','Question','Data_Value','Stratification1','StratificationCategory1']]
obesidad_recortado.head()

,YearStart,LocationAbbr,Topic,Question,Data_Value,Stratification1,StratificationCategory1
0,2011,AL,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,32.0,Total,Total
1,2011,AL,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,32.3,Male,Gender
2,2011,AL,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,31.8,Female,Gender
3,2011,AL,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,33.6,Less than high school,Education
4,2011,AL,Obesity / Weight Status,Percent of adults aged 18 years and older who ...,32.8,High school graduate,Education


In [76]:
obesidad_recortado['Question'].unique()

array(['Percent of adults aged 18 years and older who have obesity',
       'Percent of adults aged 18 years and older who have an overweight classification',
       'Percent of adults who report consuming fruit less than one time daily',
       'Percent of adults who report consuming vegetables less than one time daily',
       'Percent of adults who engage in muscle-strengthening activities on 2 or more days a week',
       'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)',
       'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week',
       'Percent of adults who achieve at least 300 minutes a week of moderate-intensity aerobic physical activity 

In [77]:
obesidad_recortado['Stratification1'].unique()

array(['Total', 'Male', 'Female', 'Less than high school',
       'High school graduate', 'Some college or technical school',
       'College graduate', '18 - 24', '25 - 34', '35 - 44', '45 - 54',
       '55 - 64', '65 or older', 'Less than $15,000', '$15,000 - $24,999',
       '$25,000 - $34,999', '$35,000 - $49,999', '$50,000 - $74,999',
       '$75,000 or greater', 'Data not reported', 'Non-Hispanic White',
       'Non-Hispanic Black', 'Hispanic', 'Asian',
       'Hawaiian/Pacific Islander', 'American Indian/Alaska Native',
       '2 or more races', 'Other'], dtype=object)

In [78]:
# 3. Transformar de formato largo a formato ancho
df_pivot = obesidad_recortado.pivot_table(
    index=['YearStart', 'LocationAbbr', 'Stratification1'],
    columns='Question',
    values='Data_Value'
).reset_index()

df_pivot.columns.name = None  # Elimina el nombre de la jerarquía de columnas

In [79]:
df_pivot.head()

,YearStart,LocationAbbr,Stratification1,Percent of adults aged 18 years and older who have an overweight classification,Percent of adults aged 18 years and older who have obesity,Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination),Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week,Percent of adults who achieve at least 300 minutes a week of moderate-intensity aerobic physical activity or 150 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination),Percent of adults who engage in muscle-strengthening activities on 2 or more days a week,Percent of adults who engage in no leisure-time physical activity,Percent of adults who report consuming fruit less than one time daily,Percent of adults who report consuming vegetables less than one time daily
0,2011,AK,"$15,000 - $24,999",33.5,30.2,51.8,17.3,32.7,28.1,24.6,44.6,26.1
1,2011,AK,"$25,000 - $34,999",38.1,32.2,57.9,20.2,38.2,24.5,25.8,45.6,24.4
2,2011,AK,"$35,000 - $49,999",42.3,24.5,55.1,21.6,34.1,31.4,25.7,37.2,19.4
3,2011,AK,"$50,000 - $74,999",41.6,27.8,59.3,26.1,32.6,37.7,19.2,39.1,16.5
4,2011,AK,"$75,000 or greater",39.9,26.9,63.2,30.8,44.8,38.0,17.1,31.9,11.8


In [80]:
df_pivot.columns

Index(['YearStart', 'LocationAbbr', 'Stratification1',
       'Percent of adults aged 18 years and older who have an overweight classification',
       'Percent of adults aged 18 years and older who have obesity',
       'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)',
       'Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week',
       'Percent of adults who achieve at least 300 minutes a week of moderate-intensity aerobic physical activity or 150 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)',
       'Percent of adults who engage in muscle-strengthening activities on 2 or more days a week',
       'Percent 

In [81]:
df_pivot.corr(numeric_only=True)['Percent of adults aged 18 years and older who have obesity'].abs().sort_values(ascending=False)[1:]

Percent of adults who engage in no leisure-time physical activity                                                                                                                                                                                        0.486281
Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week    0.485847
Percent of adults who engage in muscle-strengthening activities on 2 or more days a week                                                                                                                                                                 0.483511
Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination)                              

In [82]:
df_pivot = pd.get_dummies(df_pivot, columns=["LocationAbbr", "Stratification1"], dtype=int)

In [83]:
df_pivot.head()

,YearStart,Percent of adults aged 18 years and older who have an overweight classification,Percent of adults aged 18 years and older who have obesity,Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination),Percent of adults who achieve at least 150 minutes a week of moderate-intensity aerobic physical activity or 75 minutes a week of vigorous-intensity aerobic physical activity and engage in muscle-strengthening activities on 2 or more days a week,Percent of adults who achieve at least 300 minutes a week of moderate-intensity aerobic physical activity or 150 minutes a week of vigorous-intensity aerobic activity (or an equivalent combination),Percent of adults who engage in muscle-strengthening activities on 2 or more days a week,Percent of adults who engage in no leisure-time physical activity,Percent of adults who report consuming fruit less than one time daily,Percent of adults who report consuming vegetables less than one time daily,...,Stratification1_High school graduate,Stratification1_Hispanic,"Stratification1_Less than $15,000",Stratification1_Less than high school,Stratification1_Male,Stratification1_Non-Hispanic Black,Stratification1_Non-Hispanic White,Stratification1_Other,Stratification1_Some college or technical school,Stratification1_Total
0,2011,33.5,30.2,51.8,17.3,32.7,28.1,24.6,44.6,26.1,...,0,0,0,0,0,0,0,0,0,0
1,2011,38.1,32.2,57.9,20.2,38.2,24.5,25.8,45.6,24.4,...,0,0,0,0,0,0,0,0,0,0
2,2011,42.3,24.5,55.1,21.6,34.1,31.4,25.7,37.2,19.4,...,0,0,0,0,0,0,0,0,0,0
3,2011,41.6,27.8,59.3,26.1,32.6,37.7,19.2,39.1,16.5,...,0,0,0,0,0,0,0,0,0,0
4,2011,39.9,26.9,63.2,30.8,44.8,38.0,17.1,31.9,11.8,...,0,0,0,0,0,0,0,0,0,0


In [84]:
# Eliminar filas con NaN en cualquier columna
df_pivot = df_pivot.dropna()

# O solo en columnas específicas
# df_pivot = df_pivot.dropna(subset=['tu_columna_target'])

# Verificar que ya no hay NaNs
df_pivot.isna().sum().sum()

np.int64(0)

In [85]:
X = df_pivot.drop(['Percent of adults aged 18 years and older who have obesity'],axis=1)
y = df_pivot['Percent of adults aged 18 years and older who have obesity'].to_frame()

In [86]:
escalador = StandardScaler()
X = escalador.fit_transform(X)

In [87]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42)

In [ ]:
tree_clf = DecisionTreeRegressor(criterion='squared_error', max_depth=15, max_leaf_nodes=None, min_samples_leaf=5, min_samples_split=50, random_state=42)
tree_clf.fit(X_train, y_train)

DecisionTreeRegressor()

In [89]:
y_pred = tree_clf.predict(X_test)

In [90]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

MSE: 25.4484
RMSE: 5.0446
MAE: 3.5894
R2: 0.4398


In [71]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

def mejor_combinacion_grid_search(tipo='regressor'):
    """
    Encuentra la mejor combinación usando GridSearchCV (prueba TODAS las combinaciones).
    """
    # Definir el espacio de búsqueda
    param_grid = {
        'max_depth': [3, 5, 10, 15, 20, None],
        'min_samples_split': [2, 10, 50, 100],
        'min_samples_leaf': [1, 5, 20, 50],
        'max_leaf_nodes': [None, 8, 16, 32, 64]
    }
    
    if tipo == 'regressor':
        param_grid['criterion'] = ['squared_error', 'friedman_mse', 'absolute_error']
        model = DecisionTreeRegressor(random_state=42)
        scoring = 'r2'
    else:
        param_grid['criterion'] = ['gini', 'entropy']
        model = DecisionTreeClassifier(random_state=42)
        scoring = 'accuracy'
    
    print(f"\n{'='*80}")
    print(f"GRID SEARCH - {tipo.upper()}")
    print(f"Total de combinaciones a probar: {6 * 3 * 4 * 4 * 5}")
    print(f"{'='*80}\n")
    
    # Realizar Grid Search
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        cv=5,  # Validación cruzada con 5 folds
        n_jobs=-1,  # Usar todos los procesadores
        verbose=2
    )
    
    grid_search.fit(X_train, y_train)
    
    # Resultados
    print(f"\n{'='*80}")
    print(f"MEJOR COMBINACIÓN ENCONTRADA (Grid Search)")
    print(f"{'='*80}")
    print(f"Mejor score (CV): {grid_search.best_score_:.4f}")
    print(f"\nHiperparámetros:")
    for param, valor in grid_search.best_params_.items():
        print(f"  - {param:20}: {valor}")
    
    # Evaluar en test
    y_pred = grid_search.best_estimator_.predict(X_test)
    if tipo == 'regressor':
        test_score = r2_score(y_test, y_pred)
        print(f"\nR² en test: {test_score:.4f}")
    else:
        test_score = accuracy_score(y_test, y_pred)
        print(f"\nAccuracy en test: {test_score:.4f}")
    
    print(f"{'='*80}\n")
    
    return grid_search.best_params_, grid_search.best_estimator_


# ========== USO ==========

# Para REGRESIÓN:
mejores_params, mejor_modelo = mejor_combinacion_grid_search(tipo='regressor')

# Para CLASIFICACIÓN:
# mejores_params, mejor_modelo = mejor_combinacion_grid_search(tipo='classifier')


GRID SEARCH - REGRESSOR
Total de combinaciones a probar: 1440

Fitting 5 folds for each of 1440 candidates, totalling 7200 fits
[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samples_split=10; total time=   0.0s[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samples_split=2; total time=   0.0s

[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samples_split=50; total time=   0.0s
[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samples_split=50; total time=   0.0s
[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samples_split=100; total time=   0.0s
[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samples_split=100; total time=   0.0s
[CV] END criterion=squared_error, max_depth=3, max_leaf_nodes=None, min_samples_leaf=1, min_samp

Exception ignored in: <function ResourceTracker.__del__ at 0x70abd838e8e0>
Traceback (most recent call last):
  File "/home/ciabd14/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/ciabd14/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/ciabd14/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7903aab868e0>
Traceback (most recent call last):
  File "/home/ciabd14/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/ciabd14/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/ciabd14/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTrac